In [24]:
import os
import torch
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from torch_geometric.data import HeteroData
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# =====================================================================
# 🛠️ 路径配置
# =====================================================================
DATA_DIR = "/hy-tmp/else/data/data"
OUT_PT_PATH = "/hy-tmp/else/processed/final_hetero_data_raw.pt"

FILE_CTD_DISEASE = os.path.join(DATA_DIR, "CTD_diseases.csv")
FILE_CTD_CHEM_DIS = os.path.join(DATA_DIR, "CTD_chemicals_diseases.csv")
FILE_CTD_GENE_DIS = os.path.join(DATA_DIR, "CTD_curated_genes_diseases.csv.gz")
FILE_DRUGBANK_XML = os.path.join(DATA_DIR, "full database.xml")
FILE_ID_MAPPING = os.path.join(DATA_DIR, "id_mapping.csv") 
FILE_STRING_LINKS = os.path.join(DATA_DIR, "9606.protein.links.v12.0.txt.gz")
FILE_STRING_ALIASES = os.path.join(DATA_DIR, "9606.protein.aliases.v12.0.txt.gz")

DISEASE_KEYWORDS = [
    "Alzheimer", "Parkinson", "Huntington", "ALS",
    "schizophrenia", "depression", "bipolar",
    "dementia", "epilepsy", "autism"
]

def read_ctd_safe(file_path, target_columns):
    # 使用 sep=None 自动识别，comment='#' 跳过注释
    df = pd.read_csv(file_path, dtype=str, header=None, comment='#', sep=None, engine='python')
    
    # 动态重命名：只修改我们感兴趣的前几列，剩下的保持默认数字索引
    new_cols = {i: col for i, col in enumerate(target_columns)}
    df = df.rename(columns=new_cols)
    
    return df.fillna('')

def build_graph_from_raw():
    print("🚀 启动 MSRHGNN 异构图终极构建...")
    data = HeteroData()

    # --- 1. 疾病 & 映射字典 ---
    print("\n[1/6] 读取疾病与映射字典...")
    cols_dis = ['DiseaseName', 'DiseaseID', 'AltDiseaseIDs', 'Definition',
                'ParentIDs', 'TreeNumbers', 'ParentTreeNumbers', 'Synonyms', 'SlimMappings']
    df_dis_all = read_ctd_safe(FILE_CTD_DISEASE, cols_dis)
    mask = df_dis_all['DiseaseName'].str.contains('|'.join(DISEASE_KEYWORDS), case=False, na=False)
    df_dis = df_dis_all[mask].reset_index(drop=True)
    disease_ids = df_dis['DiseaseID'].tolist()
    disease_id_to_idx = {d: i for i, d in enumerate(disease_ids)}
    data['disease'].node_id = disease_ids

    df_map = pd.read_csv(FILE_ID_MAPPING, sep=None, engine='python', dtype=str).dropna()
    uniprot_to_entrez = dict(zip(df_map['From'].str.strip(), df_map['To'].str.strip()))

    # --- 2. DrugBank XML 桥梁 ---
    print("\n[2/6] 解析 DrugBank 建立 ID 桥梁...")
    ns = '{http://www.drugbank.ca}'
    cas_to_db, mesh_to_db, db_to_uniprots = {}, {}, {}
    context = ET.iterparse(FILE_DRUGBANK_XML, events=('end',))
    for event, elem in context:
        if elem.tag == f'{ns}drug':
            id_elem = elem.find(f'{ns}drugbank-id[@primary="true"]')
            if id_elem is not None:
                db_id = id_elem.text
                cas_elem = elem.find(f'{ns}cas-number')
                if cas_elem is not None and cas_elem.text: cas_to_db[cas_elem.text.strip()] = db_id
                uniprots = [poly.get('id') for t in (elem.find(f'{ns}targets') or []) 
                            for poly in [t.find(f'{ns}polypeptide')] if poly is not None and poly.get('id')]
                if uniprots: db_to_uniprots[db_id] = uniprots
            elem.clear()

    # --- 3. 物理边 (CTD) ---
    print("\n[3/6] 构建物理边...")
    cols_gene_dis = ['GeneSymbol', 'GeneID', 'DiseaseName', 'DiseaseID', 'DirectEvidence']
    df_gene_dis = read_ctd_safe(FILE_CTD_GENE_DIS, cols_gene_dis)
    df_gene_dis = df_gene_dis[df_gene_dis['DiseaseID'].isin(disease_ids)]
    df_gene_dis = df_gene_dis[df_gene_dis['DirectEvidence'] != '']
    gene_ids = df_gene_dis['GeneID'].unique().tolist()
    gene_id_to_idx = {g: i for i, g in enumerate(gene_ids)}
    data['gene'].node_id = gene_ids

    src_assoc = [disease_id_to_idx[r['DiseaseID']] for _, r in df_gene_dis.iterrows() if r['GeneID'] in gene_id_to_idx]
    dst_assoc = [gene_id_to_idx[r['GeneID']] for _, r in df_gene_dis.iterrows() if r['GeneID'] in gene_id_to_idx]
    data['disease', 'associated_with', 'gene'].edge_index = torch.tensor([src_assoc, dst_assoc], dtype=torch.long)

    cols_chem_dis = ['ChemicalName', 'ChemicalID', 'CasRN', 'DiseaseName', 'DiseaseID', 'DirectEvidence']
    df_chem_dis = read_ctd_safe(FILE_CTD_CHEM_DIS, cols_chem_dis)
    df_chem_dis = df_chem_dis[df_chem_dis['DiseaseID'].isin(disease_ids) & df_chem_dis['DirectEvidence'].str.contains('therapeutic', na=False)]
    
    drug_id_set = set()
    src_treats_tmp, dst_treats_tmp = [], []
    for _, row in df_chem_dis.iterrows():
        db_id = cas_to_db.get(row['CasRN'].strip()) or mesh_to_db.get(row['ChemicalID'].replace('MESH:', '').strip())
        if db_id:
            drug_id_set.add(db_id)
            src_treats_tmp.append(db_id)
            dst_treats_tmp.append(disease_id_to_idx[row['DiseaseID']])
    
    drug_ids = list(drug_id_set)
    drug_id_to_idx = {d: i for i, d in enumerate(drug_ids)}
    data['drug'].node_id = drug_ids
    data['drug', 'treats', 'disease'].edge_index = torch.tensor([[drug_id_to_idx[d] for d in src_treats_tmp], dst_treats_tmp], dtype=torch.long)

    # --- 4. 药物-靶点 (高阶) ---
    print("\n[4/6] 构建药物-靶点高阶边...")
    src_t, dst_t = [], []
    for db_id in drug_ids:
        for up_id in db_to_uniprots.get(db_id, []):
            entrez = uniprot_to_entrez.get(up_id)
            if entrez in gene_id_to_idx:
                src_t.append(drug_id_to_idx[db_id]); dst_t.append(gene_id_to_idx[entrez])
    data['drug', 'targets', 'gene'].edge_index = torch.unique(torch.tensor([src_t, dst_t], dtype=torch.long), dim=1)

    # --- 5. 疾病三网络 (H, G, T) ---
    print("\n[5/6] 构建疾病相似度三网络 (H, G, T)...")
    num_diseases = len(disease_ids)
    
    # sim_h (ParentIDs 修复)
    sim_h_mat = np.zeros((num_diseases, num_diseases))
    parents = [set(str(p).split('|')) if p else set() for p in df_dis['ParentIDs']]
    for i in range(num_diseases):
        for j in range(i+1, num_diseases):
            if parents[i] & parents[j]: sim_h_mat[i, j] = sim_h_mat[j, i] = len(parents[i] & parents[j]) / len(parents[i] | parents[j])
    idx_h = np.nonzero(sim_h_mat)
    data['disease', 'sim_h', 'disease'].edge_index = torch.tensor(np.stack(idx_h), dtype=torch.long)

    # sim_g (STRING PPI 强化 - 修复 NameError)
    print("  -> 载入 STRING 数据库构建 PPI 网络...")
    df_aliases = pd.read_csv(FILE_STRING_ALIASES, sep='\t', usecols=['#string_protein_id', 'alias', 'source'])
    df_aliases = df_aliases[df_aliases['source'].str.contains('EntrezGene', na=False, case=False)]
    ensp_to_entrez = dict(zip(df_aliases['#string_protein_id'], df_aliases['alias'])) # 定义位置修复
    
    df_links = pd.read_csv(FILE_STRING_LINKS, sep=' ')
    df_links = df_links[df_links['combined_score'] > 400]
    ppi_graph = {}
    for _, row in df_links.iterrows():
        e1, e2 = ensp_to_entrez.get(row['protein1']), ensp_to_entrez.get(row['protein2'])
        if e1 in gene_id_to_idx and e2 in gene_id_to_idx:
            idx1, idx2 = gene_id_to_idx[e1], gene_id_to_idx[e2]
            ppi_graph.setdefault(idx1, set()).add(idx2)
            ppi_graph.setdefault(idx2, set()).add(idx1)

    dis_to_expanded = {}
    for i in range(num_diseases):
        g_set = {dst_assoc[k] for k, v in enumerate(src_assoc) if v == i}
        expanded = set(g_set)
        for g in g_set: expanded.update(ppi_graph.get(g, set()))
        dis_to_expanded[i] = expanded
    
    sim_g_mat = np.zeros((num_diseases, num_diseases))
    for i in range(num_diseases):
        for j in range(i+1, num_diseases):
            if dis_to_expanded[i] & dis_to_expanded[j]:
                sim_g_mat[i, j] = sim_g_mat[j, i] = len(dis_to_expanded[i] & dis_to_expanded[j]) / len(dis_to_expanded[i] | dis_to_expanded[j])
    idx_g = np.nonzero(sim_g_mat)
    data['disease', 'sim_g', 'disease'].edge_index = torch.tensor(np.stack(idx_g), dtype=torch.long)

    # sim_t (临床文本强化)
    texts = (df_dis['DiseaseName'] + " " + df_dis['Definition'] + " " + df_dis['Synonyms']).tolist()
    X = TfidfVectorizer(stop_words='english', max_features=1024).fit_transform(texts)
    nbrs = NearestNeighbors(n_neighbors=min(6, X.shape[0]), metric='cosine').fit(X)
    dist, idx = nbrs.kneighbors(X)
    src_t, dst_t, val_t = [], [], []
    for i in range(len(idx)):
        for j, d in zip(idx[i][1:], dist[i][1:]):
            src_t.append(i); dst_t.append(j); val_t.append(1 - d)
    data['disease', 'sim_t', 'disease'].edge_index = torch.tensor([src_t, dst_t], dtype=torch.long)
    data['disease', 'sim_t', 'disease'].edge_attr = torch.tensor(val_t, dtype=torch.float).unsqueeze(1)

    # --- 6. 保存 ---
    data['drug'].x = torch.randn((len(drug_ids), 128))
    data['disease'].x = torch.tensor(X.toarray(), dtype=torch.float)
    data['gene'].x = torch.randn((len(gene_ids), 128))
    torch.save(data, OUT_PT_PATH)
    print(f"\n🎉 构建完成！图数据已保存至 {OUT_PT_PATH}\n", data)

if __name__ == "__main__":
    build_graph_from_raw()

🚀 启动 MSRHGNN 异构图终极构建...

[1/6] 读取疾病与映射字典...

[2/6] 解析 DrugBank 建立 ID 桥梁...

[3/6] 构建物理边...

[4/6] 构建药物-靶点高阶边...

[5/6] 构建疾病相似度三网络 (H, G, T)...
  -> 载入 STRING 数据库构建 PPI 网络...

🎉 构建完成！图数据已保存至 /hy-tmp/else/processed/final_hetero_data_raw.pt
 HeteroData(
  disease={
    node_id=[247],
    x=[247, 1024],
  },
  gene={
    node_id=[927],
    x=[927, 128],
  },
  drug={
    node_id=[265],
    x=[265, 128],
  },
  (disease, associated_with, gene)={ edge_index=[2, 1168] },
  (drug, treats, disease)={ edge_index=[2, 510] },
  (drug, targets, gene)={ edge_index=[2, 618] },
  (disease, sim_h, disease)={ edge_index=[2, 3498] },
  (disease, sim_g, disease)={ edge_index=[2, 352] },
  (disease, sim_t, disease)={
    edge_index=[2, 1235],
    edge_attr=[1235, 1],
  }
)


In [25]:
import torch

PT_PATH = "/hy-tmp/else/processed/final_hetero_data_raw.pt"

def inspect_hetero_data(pt_path):
    print("🚀 加载数据...")
    data = torch.load(pt_path)

    print("\n" + "="*60)
    print("📊 基本信息")
    print("="*60)

    print(f"节点类型: {list(data.node_types)}")
    print(f"边类型: {list(data.edge_types)}")

    # ============================================================
    # 🧬 节点信息
    # ============================================================
    print("\n" + "="*60)
    print("🧬 节点详情")
    print("="*60)

    total_nodes = 0

    for ntype in data.node_types:
        node_data = data[ntype]

        num_nodes = node_data.num_nodes
        total_nodes += num_nodes

        print(f"\n🔹 节点类型: {ntype}")
        print(f"  数量: {num_nodes}")

        # 特征
        if hasattr(node_data, 'x'):
            print(f"  特征维度: {tuple(node_data.x.shape)}")
        else:
            print("  ⚠️ 无特征")

        # node_id
        if hasattr(node_data, 'node_id'):
            ids = node_data.node_id
            print(f"  ID示例: {ids[:5]}")
        else:
            print("  ⚠️ 无 node_id")

    print(f"\n🧮 总节点数: {total_nodes}")

    # ============================================================
    # 🔗 边信息
    # ============================================================
    print("\n" + "="*60)
    print("🔗 边详情")
    print("="*60)

    total_edges = 0

    for etype in data.edge_types:
        edge_data = data[etype]

        num_edges = edge_data.edge_index.shape[1]
        total_edges += num_edges

        print(f"\n🔸 边类型: {etype}")
        print(f"  边数量: {num_edges}")

        # 是否有权重
        if hasattr(edge_data, 'edge_attr'):
            print(f"  权重维度: {tuple(edge_data.edge_attr.shape)}")
        else:
            print("  无权重")

        # 示例边
        ei = edge_data.edge_index
        print("  示例边:")
        for i in range(min(5, num_edges)):
            print(f"    {ei[0, i]} -> {ei[1, i]}")

    print(f"\n🧮 总边数: {total_edges}")

    # ============================================================
    # 📈 密度 & 稀疏性
    # ============================================================
    print("\n" + "="*60)
    print("📈 图统计")
    print("="*60)

    for etype in data.edge_types:
        src_type, _, dst_type = etype
        num_src = data[src_type].num_nodes
        num_dst = data[dst_type].num_nodes
        num_edges = data[etype].edge_index.shape[1]

        density = num_edges / (num_src * num_dst)

        print(f"{etype}:")
        print(f"  密度: {density:.6e}")

    print("\n🎉 检查完成！")

# ============================================================
if __name__ == "__main__":
    inspect_hetero_data(PT_PATH)

🚀 加载数据...

📊 基本信息
节点类型: ['disease', 'gene', 'drug']
边类型: [('disease', 'associated_with', 'gene'), ('drug', 'treats', 'disease'), ('drug', 'targets', 'gene'), ('disease', 'sim_h', 'disease'), ('disease', 'sim_g', 'disease'), ('disease', 'sim_t', 'disease')]

🧬 节点详情

🔹 节点类型: disease
  数量: 247
  特征维度: (247, 1024)
  ID示例: ['OMIM:611812', 'MESH:D015526', 'MESH:C537052', 'MESH:C537057', 'MESH:C536588']

🔹 节点类型: gene
  数量: 927
  特征维度: (927, 128)
  ID示例: ['1', '2', '20', '10347', '10351']

🔹 节点类型: drug
  数量: 265
  特征维度: (265, 128)
  ID示例: ['DB00829', 'DB11699', 'DB01104', 'DB00850', 'DB03205']

🧮 总节点数: 1439

🔗 边详情

🔸 边类型: ('disease', 'associated_with', 'gene')
  边数量: 1168
  无权重
  示例边:
    225 -> 0
    6 -> 1
    39 -> 2
    6 -> 3
    20 -> 3

🔸 边类型: ('drug', 'treats', 'disease')
  边数量: 510
  无权重
  示例边:
    220 -> 6
    101 -> 182
    94 -> 225
    226 -> 210
    236 -> 211

🔸 边类型: ('drug', 'targets', 'gene')
  边数量: 618
  无权重
  示例边:
    0 -> 362
    2 -> 641
    2 -> 781
    2 -> 782
    3 -> 